# TirraMind Phase 50 — Price Features + Residual Returns

**Zero config. Just attach the two datasets and click Run All.**

## Setup (one-time, already done if running via `kaggle_launch.py`)

- **Dataset 1:** `tirramind-data` → contains `pipeline.db`
- **Dataset 2:** `tirramind-code` → contains `agent/` + `scripts/` (auto-uploaded by `kaggle_launch.py`)
- **Accelerator:** GPU T4 x1 (Settings → Accelerator)
- W&B optional: add `WANDB_API_KEY` Kaggle Secret for live loss monitoring

In [ ]:
import subprocess
import sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *args], check=True)

# torch-geometric 2.7+ does NOT require torch-scatter/torch-sparse.
# Installing those from pyg.org wheels causes CUDA architecture mismatches on Kaggle T4.
# PyG 2.7 falls back to native PyTorch sparse ops automatically.
pip("torch-geometric==2.7.0")
pip("tqdm", "rich", "wandb")

import torch
import torch_geometric
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA avail   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"PyG          : {torch_geometric.__version__}")
print("Install OK.")

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ── CODE: try git clone with token, fall back to tirramind-code dataset ──────
_code_loaded = False

try:
    from kaggle_secrets import UserSecretsClient
    _token = UserSecretsClient().get_secret("tirramind_token")
    _repo_url = f"https://{_token}@github.com/savabs/tirramind.git"
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    subprocess.run(["git", "clone", "--depth=1", _repo_url, str(WORK_DIR)],
                   check=True, capture_output=True)
    print("✓ Code: cloned from GitHub (tirramind_token secret)")
    _code_loaded = True
except Exception as _e:
    print(f"  Git clone skipped ({_e.__class__.__name__}) — using tirramind-code dataset")

if not _code_loaded:
    def find_code_root(root="/kaggle/input"):
        for dirpath, dirs, _ in os.walk(root):
            if {"agent", "scripts"}.issubset(set(dirs)):
                return Path(dirpath)
        return None
    _code_root = find_code_root()
    assert _code_root is not None, (
        "CODE NOT FOUND.\n"
        "Fix: run  python scripts/kaggle_launch.py  locally — it uploads tirramind-code automatically.\n"
        "Or add tirramind_token Kaggle Secret for GitHub clone."
    )
    for _name in ("agent", "scripts"):
        _dst = WORK_DIR / _name
        if _dst.exists():
            shutil.rmtree(_dst)
        shutil.copytree(_code_root / _name, _dst)
    print(f"✓ Code: loaded from dataset at {_code_root}")

# ── DATA: pipeline.db from tirramind-data dataset ────────────────────────────
def find_data_root(root="/kaggle/input"):
    for dirpath, dirs, files in os.walk(root):
        if "pipeline.db" in set(files):
            return Path(dirpath)
    return None

_data_root = find_data_root()
assert _data_root is not None, (
    "DATA NOT FOUND.\n"
    "Fix: attach 'tirramind-data' dataset in the right panel → Data → Add Dataset."
)
pipeline_dir = WORK_DIR / ".tirra_pipeline"
pipeline_dir.mkdir(exist_ok=True)
shutil.copy2(_data_root / "pipeline.db", pipeline_dir / "pipeline.db")
print(f"✓ Data: pipeline.db ({(pipeline_dir / 'pipeline.db').stat().st_size // 1_000_000} MB)")

# ── Patch pipeline __init__ for lazy APScheduler import ──────────────────────
(WORK_DIR / "agent" / "pipeline" / "__init__.py").write_text(
    '"""TirraMind — Pipeline Layer."""\n\n'
    'from agent.pipeline.storage_backend import PostgresBackend, SQLiteBackend, StorageBackend\n'
    'from agent.pipeline.store import PipelineStore\n\n'
    '__all__ = ["PipelineStore", "PipelineScheduler", "StorageBackend", "SQLiteBackend", "PostgresBackend"]\n\n'
    'def __getattr__(name):\n'
    '    if name == "PipelineScheduler":\n'
    '        from agent.pipeline.scheduler import PipelineScheduler\n'
    '        return PipelineScheduler\n'
    '    raise AttributeError(f"module {__name__!r} has no attribute {name!r}")\n',
    encoding="utf-8",
)
print("✓ Patched pipeline __init__")
print("\nSetup complete — ready to train.")

In [ ]:
import torch
import torch_geometric

# ── GPU compatibility check ───────────────────────────────────────────────────
# Kaggle randomly assigns T4 (sm_75) or P100 (sm_60).
# PyTorch 2.5+ dropped support for sm_60 → fall back to CPU.
# Adaptive window count so training always fits within 12h:
#   GPU (sm_70+) : 200 windows × ~2s  = ~7  min/epoch × 30 = ~3.5h  ✓
#   CPU fallback : 80  windows × ~13s = ~17 min/epoch × 30 = ~8.5h  ✓

DEVICE = "cpu"
MAX_WINDOWS = 80  # safe default for CPU

if torch.cuda.is_available():
    cap_major, cap_minor = torch.cuda.get_device_capability(0)
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU          : {gpu_name}  (sm_{cap_major}{cap_minor})")
    if cap_major >= 7:
        DEVICE = "cuda"
        MAX_WINDOWS = 200
        print(f"GPU OK       : sm_{cap_major}{cap_minor} compatible — full 200 windows/epoch")
    else:
        DEVICE = "cpu"
        MAX_WINDOWS = 80
        print(f"⚠ GPU SKIP   : {gpu_name} sm_{cap_major}{cap_minor} incompatible with PyTorch {torch.__version__}")
        print(f"               CPU fallback — reduced to 80 windows/epoch to fit 12h limit")
else:
    print("GPU          : none (CPU mode) — 80 windows/epoch")

print(f"PyTorch      : {torch.__version__}")
print(f"PyG          : {torch_geometric.__version__}")
print(f"Device       : {DEVICE}  |  max_windows: {MAX_WINDOWS}")

In [ ]:
from pathlib import Path
WORK_DIR = Path("/kaggle/working/tirramind_v1")
assert (WORK_DIR / ".tirra_pipeline/pipeline.db").exists()
for f in ["agent/models/gnn/graph_builder.py", "agent/models/gnn/het_tgn.py", "agent/models/gnn/trainer.py", "scripts/retrain_gnn.py"]:
    assert (WORK_DIR / f).exists(), f"Missing: {f}"
print("All checks passed. Starting Phase 50 training from epoch 0.")

In [ ]:
import subprocess
import sys
import os
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")

# ── Checkpoint dir goes directly inside /kaggle/working/ so Kaggle always
#    captures every completed epoch even if the session times out mid-run.
CKPT_DIR = Path("/kaggle/working/phase50_ckpts")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE      = globals().get("DEVICE", "cpu")
MAX_WINDOWS = globals().get("MAX_WINDOWS", 80)

# W&B setup
_wandb_project = None
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = _wandb_key
    _wandb_project = "tirramind"
    print("W&B enabled")
except Exception as _e:
    print(f"W&B disabled ({_e.__class__.__name__})")

print(f"Device: {DEVICE}  |  max_windows: {MAX_WINDOWS}")
print(f"Checkpoints → {CKPT_DIR}  (captured even on timeout)")
print("Phase 50: hidden=128, heads=4, price features, residual returns, ListNet + auto-tune")

cmd = [
    sys.executable, "scripts/retrain_gnn.py",
    "--epochs",              "30",
    "--hidden-dim",          "128",
    "--num-layers",          "2",
    "--num-heads",           "4",
    "--lr",                  "1e-3",
    "--backup",
    "--window-size",         "604800",
    "--gdelt-frac",          "0.05",
    "--max-windows",         str(MAX_WINDOWS),
    "--auto-tune",
    "--listnet",
    "--return-weight",       "3.0",
    "--return-log-var-max",  "0.0",
    "--direction-loss",
    "--residual-returns",
    "--device",              DEVICE,
    "--skip-eval",
    "--checkpoint-dir",      str(CKPT_DIR),
    "--model-out",           ".tirra_pipeline/gnn_model_phase50.pt",
]

if _wandb_project:
    cmd += [
        "--wandb-project", _wandb_project,
        "--wandb-run",     "phase50-ep1-30",
        "--wandb-tags",    "phase50,price-features,residual-returns",
    ]

print("Running:", " ".join(cmd))
print("-" * 70)

process = subprocess.Popen(
    cmd, cwd=str(WORK_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Training failed: exit {return_code}")
print("\nPhase 50 training completed.")

In [ ]:
import shutil
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")
# Checkpoints were written directly to /kaggle/working/phase50_ckpts/
# — already captured by Kaggle. Just confirm and list them.
CKPT_DIR = Path("/kaggle/working/phase50_ckpts")
OUT_DIR  = Path("/kaggle/working")

# Final model (written by retrain_gnn.py --model-out)
final_model = WORK_DIR / ".tirra_pipeline" / "gnn_model_phase50.pt"
if final_model.exists():
    shutil.copy2(final_model, OUT_DIR / "gnn_model_phase50.pt")
    print(f"gnn_model_phase50.pt → /kaggle/working/ ({final_model.stat().st_size / 1_000_000:.1f} MB)")
else:
    print("⚠ gnn_model_phase50.pt not found (training may not have completed all epochs)")

# List captured checkpoints
ckpts = sorted(CKPT_DIR.glob("epoch_*.pt"))
print(f"\nCheckpoints in /kaggle/working/phase50_ckpts/ ({len(ckpts)} epochs):")
for ckpt in ckpts:
    print(f"  {ckpt.name}  ({ckpt.stat().st_size / 1_000_000:.1f} MB)")

print("\nAll files above are available in the Output tab for download.")

In [ ]:
# Optional: run backtest directly on Kaggle to see IC before downloading
import subprocess, sys, shutil
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")
MODEL = WORK_DIR / ".tirra_pipeline" / "gnn_model_phase50.pt"
assert MODEL.exists(), "Model not found — training may not have completed"

# Symlink to expected name
link = WORK_DIR / ".tirra_pipeline" / "gnn_model.pt"
if link.exists():
    link.unlink()
shutil.copy2(MODEL, link)

result = subprocess.run(
    [sys.executable, str(WORK_DIR / "scripts/phase40_gnn_backtest.py"), "--out", ".tirra_pipeline/ic_results_phase50.json"],
    cwd=str(WORK_DIR), text=True
)

if result.returncode == 0:
    print("\nBacktest complete. Check output above for IC results.")
else:
    print("Backtest FAILED")